In [1]:
import pandas as pd
import os
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from extractor import MediapipeExtractor
from sklearn import preprocessing

from scipy.signal import savgol_filter

In [ ]:
names = os.listdir("data/mediapipe/csv")
names = sorted(names)

dataframes = []
for name in names:
    dataframe = pd.read_csv(f"data/mediapipe/csv/{name}")
    dataframe = pd.DataFrame(savgol_filter(dataframe, 20, 2, axis=0),
                                columns=dataframe.columns,
                                index=dataframe.index)
    dataframes.append(dataframe)

mp_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
names = os.listdir("data/mocap/csv")
names = sorted(names)

dataframes = []
for name in names:
    dataframes.append(pd.read_csv(f"data/mocap/csv/{name}"))

quat_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
quat_segments = {
    "left_hand": {
        "CC_Base_L_Mid1",
        "CC_Base_L_Mid2",
        "CC_Base_L_Mid3",
        "CC_Base_L_Index1",
        "CC_Base_L_Index2",
        "CC_Base_L_Index3",
        "CC_Base_L_Ring1",
        "CC_Base_L_Ring2",
        "CC_Base_L_Ring3",
        "CC_Base_L_Pinky1",
        "CC_Base_L_Pinky2",
        "CC_Base_L_Pinky3",
        "CC_Base_L_Thumb1",
        "CC_Base_L_Thumb2",
        "CC_Base_L_Thumb3",
        "CC_Base_L_Clavicle",
        "CC_Base_L_Upperarm",
        "CC_Base_L_Forearm",
        "CC_Base_L_Hand",
        },
    "right_hand":{
        "CC_Base_R_Mid1",
        "CC_Base_R_Mid2",
        "CC_Base_R_Mid3",
        "CC_Base_R_Ring1",
        "CC_Base_R_Ring2",
        "CC_Base_R_Ring3",
        "CC_Base_R_Thumb1",
        "CC_Base_R_Thumb2",
        "CC_Base_R_Thumb3",
        "CC_Base_R_Index1",
        "CC_Base_R_Index2",
        "CC_Base_R_Index3",
        "CC_Base_R_Pinky1",
        "CC_Base_R_Pinky2",
        "CC_Base_R_Pinky3",
        "CC_Base_R_Clavicle",
        "CC_Base_R_Upperarm",
        "CC_Base_R_Forearm",
        "CC_Base_R_Hand",
        },
    "body":{
        "CC_Base_Hip",
        "CC_Base_FacialBone",
        "CC_Base_NeckTwist02",
        "CC_Base_NeckTwist01",
        "CC_Base_Pelvis",
        "CC_Base_Waist",
        "CC_Base_Spine01",
        "CC_Base_Spine02",
        "CC_Base_BoneRoot",
        # !Ноги
        "CC_Base_R_Thigh",
        "CC_Base_R_Calf",
        "CC_Base_R_Foot",
        "CC_Base_L_Thigh",
        "CC_Base_L_Calf",
        "CC_Base_L_Foot",
        }
    }

In [ ]:
def get_columns_names(segments):
    col = []
    for item in segments:
        col.append(item+".Q0")
        col.append(item+".Q1")
        col.append(item+".Q2")
        col.append(item+".Q3")
    return col

In [ ]:
from sklearn.model_selection import GridSearchCV


In [ ]:
models = []

for segment, columns in quat_segments.items():
    columns = get_columns_names(columns)
    model = KNeighborsRegressor()
    models.append({
        "segment": segment,
        "model": model,
        "columns": columns
        "param_grid": {'n_neighbors': range(1, 100, 1)}
    })

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, axs = plt.subplots(len(models), 1, figsize=(5, 5*len(models)))
if len(models) == 1:
    axs = [axs]
for model in models:
	grid = GridSearchCV(model["model"], cv=5)
	grid.fit(mp_df, quat_df[model["columns"]])
	test_scores = grid.cv_results_["mean_test_score"]
	train_scores = grid.cv_results_["mean_train_score"]
	params = grid.cv_results_['params']
	axs[models.index(model)].plot(params, train_scores, label='Train Score')
	axs[models.index(model)].plot(params, test_scores, label='Test Score')
	axs[models.index(model)].set_xlabel('Params')
	axs[models.index(model)].set_ylabel('Score')
	axs[models.index(model)].set_title('График функции потерь на тесте и трейне')


In [ ]:
df = pd.read_csv("data/outputs/out_mp.csv")
#df = pd.read_csv("data/mediapipe/csv/Armature.008|standing_discuss_m_270744.csv")

In [ ]:
df = pd.DataFrame(savgol_filter(df, 20, 2, axis=0),
                                columns=df.columns,
                                index=df.index)

In [ ]:
motion = df.to_numpy()

In [ ]:
motion.shape

In [ ]:
mp_vectors = mp_df.to_numpy()

In [ ]:


regr = MLPRegressor(hidden_layer_sizes=(100, 100), random_state=1, max_iter=5000).fit(mp_vectors, quat_df)
out_frames_body = pd.DataFrame(regr.predict(motion), columns=quat_df.columns)

regr = KNeighborsRegressor(n_neighbors=best_n_neighbors, weights="distance").fit(mp_vectors, quat_vectors_wrist)
out_frames_right_wrist = pd.DataFrame(regr.predict(motion), columns=right_wrist_col+left_wrist_col)

#regr = KNeighborsRegressor(n_neighbors=32).fit(mp_vectors, quat_vectors_left_wrist)
#out_frames_left_wrist = pd.DataFrame(regr.predict(motion), columns=left_wrist_col)

out_frames_body = out_frames_body.drop(columns=wrist_col)
out_df = pd.concat([out_frames_body, out_frames_right_wrist], axis=1)


In [ ]:
out_df = pd.DataFrame(savgol_filter(out_df, 10, 2, axis=0),
                                columns=out_df.columns,
                                index=out_df.index)

In [ ]:
#out_df = out_df.rolling(15).mean().dropna(ignore_index=True)

In [ ]:
out_df.to_csv("data/outputs/motion.csv", index=False)